## Accessing and analysing data from the Perovskite Tandem Database
This notebook gives a short demonstration for how to access and query the perovskite Tandem database in NOMAD.

In order to access NOMAD entries, you first need to select a URL of the relevant NOMAD installation and provide a generated API token for authorization. Below, a URL for the central example Oasis is already selected. API token can be generated in GUI by a logged-in user; in case of example OASIS got to the [APIs page](https://nomad-lab.eu/oasis/gui/analyze/apis), under `App token` select desired expiration date and copy the token via the icon next to the right.

The token should be stored in a `.env` file in the project; this file is not tracked by github. Create it or copy it from the example file `.env.example`, then replace the placeholder with the actual token (without any quotation marks or other symbols).

In [1]:
import json
import os
import time

# import matplotlib.pyplot as plt
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

url = "https://nomad-lab.eu/oasis/backend/api/v1"
API_TOKEN = os.environ["API_TOKEN"]

### Read in a specific entry from the perovskite tandem database

In [2]:
# Read a single entry you have access to (public, created by you or shared with you)
# and export it to a JSON file. The file is selected by the entry ID

entry_id_to_export="--9Q9v_0LGFevnjJo8_PjTpwUWQr"

def load_entry(url: str, token: str, entry_id: str) -> list[dict]:
    query = {
        "required": {
            "data": "*",
        },
        "owner": "visible",
        "query": {
            "entry_id": entry_id,
            "entry_type": "PerovskiteTandemSolarCell",
        },
        "pagination": {"page_size": 50},
    }

    linked_data = []

    while True:
        response = requests.post(
            f"{url}/entries/archive/query",
            headers={"Authorization": f"Bearer {token}"},
            json=query,
        )
        response.raise_for_status()
        response_json = response.json()
        linked_data.extend(response_json["data"])

        next_value = response_json["pagination"].get("next_page_after_value")
        if not next_value:
            break
        query["pagination"]["page_after_value"] = next_value

    return linked_data

full_data = load_entry(url, API_TOKEN, entry_id_to_export)

data = full_data[0]['archive']
if "m_ref_archives" in data:
    del data["m_ref_archives"]
json_data = json.dumps(data, indent=2)

with open("result.json", "w") as f:
    f.write(json_data)

print(f"### The entry with id {entry_id_to_export} has been saved into result.json ###")

### The entry with id --9Q9v_0LGFevnjJo8_PjTpwUWQr has been saved into result.json ###


### Read in all datafiles in the perovsktie tandem database

In [8]:
def load_all_entries(url: str, token: str) -> list[dict]:
    query = {
        "required": {
            "data": "*",
        },
        "owner": "visible",
        "query": {
            "entry_type": "PerovskiteTandemSolarCell",
        },
        "pagination": {"page_size": 500},
    }

    linked_data = []

    while True:
        response = requests.post(
            f"{url}/entries/archive/query",
            headers={"Authorization": f"Bearer {token}"},
            json=query,
        )
        response.raise_for_status()
        response_json = response.json()
        linked_data.extend(response_json["data"])

        next_value = response_json["pagination"].get("next_page_after_value")
        if not next_value:
            break
        query["pagination"]["page_after_value"] = next_value
        time.sleep(3)  # Add a delay due to server rate limiting

    return linked_data

time.sleep(5)  # Add a delay due to server rate limiting
full_data = load_all_entries(url, API_TOKEN)
data = [full_data[i]['archive'] for i in range(len(full_data))]
for i in range(len(data)):
    if "m_ref_archives" in data[i]:
        del data[i]["m_ref_archives"]

print(f"### Found {len(data)} tandem solar cell entries ###")
# data = full_data[0]['archive']
# if "m_ref_archives" in data:
#     del data["m_ref_archives"]
# json_data = json.dumps(data, indent=2)


### Found 638 tandem solar cell entries ###


#### Extract selected parameters and convert them into a pandas dataframe

### Query for all 2-terminal perovskite-perovskite devices in the database